In [ ]:
##EDA AND FEATURE ENGINEERING

In [ ]:
#EDA

In [ ]:
from config import DATABASE_NAME, BUCKET, REGION, S3_STAGING_DIR

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlalchemy
from pyathena import connect
from matplotlib.ticker import MaxNLocator


# Project constants
region = REGION   # change if needed
database_name = DATABASE_NAME
bucket =  BUCKET
s3_staging_dir=S3_STAGING_DIR


s3_staging_dir = s3_staging_dir=s3_staging_dir

conn = connect(
    region_name=region,
    s3_staging_dir=S3_STAGING_DIR
)


In [ ]:
##Verify Athena Tables
pd.read_sql(f"SHOW TABLES IN {database_name}", conn)

In [ ]:
pd.read_sql(
    f"""
    SELECT MIN(date) AS min_date, MAX(date) AS max_date
    FROM retail_forecasting.train_sales
    """,
    conn
)


In [ ]:
'''We will do EDA in 3 layers:

Individual tables (sanity + patterns)

Time behavior (seasonality, trends)

Cross-table insights (before merging)'''


In [ ]:
##EDA for train.csv

In [ ]:
#QUICK CHECK

query = f"""
SELECT date, store_nbr, family, sales, onpromotion
FROM {database_name}.train_sales
WHERE CAST(date AS DATE) BETWEEN DATE '2014-01-01' AND DATE '2017-06-30'
"""

# Read SQL query into pandas DataFrame
train_df = pd.read_sql(query, conn)

# Convert date column to datetime in pandas
train_df['date'] = pd.to_datetime(train_df['date'])

# Preview the DataFrame
print(train_df.head())



In [ ]:
#Basic info
print("================Basic Info=========================")
print(train_df.info())

# First few rows
print("===================First 5 rows=============================")
print(train_df.head())

# Summary statistics
print("===================Summary Statistics====================================")
print(train_df.describe())

# Check for missing values
print("===================Check for missing values=============================")
print(train_df.isnull().sum())

In [ ]:
##Analyze the target variable sales

In [ ]:
# Histogram of sales
plt.figure(figsize=(10,5))
sns.histplot(train_df['sales'], bins=50, kde=True)
plt.title('Distribution of Sales')
plt.xlabel('Sales')
plt.ylabel('Frequency')
plt.show()

# Boxplot to see outliers
plt.figure(figsize=(10,5))
sns.boxplot(x='sales', data=train_df)
plt.title('Sales Boxplot')
plt.show()


In [ ]:
##Time based Analysis

In [ ]:
# Sales over time (total per day)
daily_sales = train_df.groupby('date')['sales'].sum().reset_index()

plt.figure(figsize=(15,5))
sns.lineplot(data=daily_sales, x='date', y='sales')
plt.title('Total Daily Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.show()


In [ ]:
# weekly or monthly trends:
train_df['month'] = train_df['date'].dt.to_period('M')
monthly_sales = train_df.groupby('month')['sales'].sum().reset_index()

plt.figure(figsize=(12,5))
sns.barplot(data=monthly_sales, x='month', y='sales', color='skyblue')
plt.title('Monthly Sales')
plt.xticks(rotation=45)
plt.show()

In [ ]:
##Sales by store and family

In [ ]:
# Total sales per store
store_sales = train_df.groupby('store_nbr')['sales'].sum().reset_index().sort_values(by='sales', ascending=False)
print(store_sales.head())

# Total sales per family
family_sales = train_df.groupby('family')['sales'].sum().reset_index().sort_values(by='sales', ascending=False)
print(family_sales.head())


In [ ]:
# Top 10 families
top_families = family_sales.head(10)
plt.figure(figsize=(12,5))
sns.barplot(data=top_families, x='family', y='sales', palette='viridis')
plt.title('Top 10 Product Families by Sales')
plt.xticks(rotation=45)
plt.show()


In [ ]:
##Promotions analysis

In [ ]:
#How many sales were on promotion?
family_promo_ratio = (
    train_df.groupby('family')
    .apply(lambda x: x.loc[x['onpromotion'] > 0, 'sales'].sum() / x['sales'].sum())
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name='promo_sales_ratio')
)

plt.figure(figsize=(8,4))
sns.barplot(data=family_promo_ratio, x='family', y='promo_sales_ratio')
plt.title('Top 10 Families by Promotion Dependency')
plt.ylabel('Proportion of Sales on Promotion')
plt.xticks(rotation=45)
plt.show()



In [ ]:
##Correlation check

In [ ]:
# Only numeric columns
numeric_cols = ['sales', 'onpromotion']
corr = train_df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


In [ ]:
#Calculate 30-Day Rolling Mean
daily_sales["rolling_30"] = daily_sales["sales"].rolling(window=30).mean()
plt.figure(figsize=(12,5))

plt.plot(daily_sales["date"],
         daily_sales["sales"],
         alpha=0.4,
         color="dimgray",
         label="Daily Sales")

plt.plot(daily_sales["date"],
         daily_sales["rolling_30"],
         color="crimson",
         linewidth=2,
         label="30-Day Rolling Mean")

plt.title("Daily Sales with 30-Day Rolling Mean")
plt.xlabel("Date")
plt.ylabel("Total Sales")
plt.legend()

plt.show()


In [ ]:
##EDA for Stores.csv

In [ ]:
#Read and do operations from data store in DataLake
query = f"""
SELECT *
FROM {database_name}.stores
LIMIT 10
"""
stores_df = pd.read_sql(query, conn)
stores_df


In [ ]:
##Row count & uniqueness
query = f"""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT store_nbr) AS unique_stores
FROM {database_name}.stores
"""
pd.read_sql(query, conn)


In [ ]:
#Missing values check
query = f"""
SELECT
    SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) AS city_nulls,
    SUM(CASE WHEN state IS NULL THEN 1 ELSE 0 END) AS state_nulls,
    SUM(CASE WHEN type IS NULL THEN 1 ELSE 0 END) AS type_nulls,
    SUM(CASE WHEN cluster IS NULL THEN 1 ELSE 0 END) AS cluster_nulls
FROM {database_name}.stores
"""
pd.read_sql(query, conn)


In [ ]:
#Categorical distributions
query = f"""
SELECT type, COUNT(*) AS store_count
FROM {database_name}.stores
GROUP BY type
ORDER BY store_count DESC
"""
type_df = pd.read_sql(query, conn)
type_df


In [ ]:
#BarPlot for storetype vs storecount 
ax = type_df.plot(kind='bar', x='type', y='store_count', title='Store Type Distribution')
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
plt.show()


In [ ]:
#Cluster distribution
query = f"""
SELECT cluster, COUNT(*) AS store_count
FROM {database_name}.stores
GROUP BY cluster
ORDER BY cluster
"""
cluster_df = pd.read_sql(query, conn)
cluster_df


In [ ]:
#State-level distribution
query = f"""
SELECT state, COUNT(*) AS store_count
FROM {database_name}.stores
GROUP BY state
ORDER BY store_count DESC
"""
state_df = pd.read_sql(query, conn)
state_df.head(10)

In [ ]:
##EDA for transactions.csv

"
transactions = total number of customer transactions per store per day
This is:
- Time-varying
- Store-level
- Strong proxy for footfall / demand

In [ ]:
query = f"""
SELECT *
FROM {database_name}.transactions
LIMIT 10
"""
transactions_df = pd.read_sql(query, conn)

transactions_df['date'] = pd.to_datetime(transactions_df['date'])
transactions_df

In [ ]:
#Statistical Analysis
transactions_df.describe()

In [ ]:
transactions_df.info()

In [ ]:
transactions_df.shape

In [ ]:
#Find any missing values present
transactions_df.isna().sum()

In [ ]:
#How many stores have transaction data
transactions_df['store_nbr'].nunique()

In [ ]:
transactions_df['date'].value_counts()

In [ ]:
query = """
SELECT 
    MIN(date) AS min_date,
    MAX(date) AS max_date
FROM retail_forecasting.transactions
"""

pd.read_sql(query, conn)

In [ ]:
#Total transactions per day
query = """
SELECT 
    date,
    SUM(transactions) AS total_transactions
FROM retail_forecasting.transactions
GROUP BY date
ORDER BY date
"""

daily_total = pd.read_sql(query, conn)
daily_total.head()

plt.figure(figsize=(12,5))
plt.plot(pd.to_datetime(daily_total["date"]),
         daily_total["total_transactions"])
plt.title("Total Daily Transactions")
plt.show()

In [ ]:
#Top 10 Stores by Average Transactions
query = """
SELECT 
    store_nbr,
    AVG(transactions) AS avg_transactions
FROM retail_forecasting.transactions
GROUP BY store_nbr
ORDER BY avg_transactions DESC
LIMIT 10
"""

top10_stores = pd.read_sql(query, conn)

plt.figure(figsize=(10,5))
plt.bar(top10_stores["store_nbr"].astype(str),
        top10_stores["avg_transactions"])

plt.title("Top 10 Stores by Average Transactions")
plt.xlabel("Store Number")
plt.ylabel("Average Transactions")
plt.xticks(rotation=45)
plt.show()


In [ ]:
#Weekly Average Transactions
query = """
SELECT 
    day_of_week(date_parse(date, '%Y-%m-%d')) AS weekday_num,
    AVG(transactions) AS avg_transactions
FROM retail_forecasting.transactions
GROUP BY 1
ORDER BY 1
"""

weekly_avg = pd.read_sql(query, conn)

# Athena: 1=Sunday ... 7=Saturday
weekday_map = {
    1: "Sunday",
    2: "Monday",
    3: "Tuesday",
    4: "Wednesday",
    5: "Thursday",
    6: "Friday",
    7: "Saturday"
}

weekly_avg["weekday"] = weekly_avg["weekday_num"].map(weekday_map)

# Reorder Monday → Sunday
order = ["Monday", "Tuesday", "Wednesday",
         "Thursday", "Friday", "Saturday", "Sunday"]

weekly_avg = weekly_avg.set_index("weekday").loc[order].reset_index()

plt.figure(figsize=(8,5))
plt.bar(weekly_avg["weekday"],
        weekly_avg["avg_transactions"],
       color = "purple")

plt.title("Average Transactions by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average Transactions")
plt.show()


In [ ]:
#Monthly Average Transactions (Jan → Dec)
query = """
SELECT 
    month(date_parse(date, '%Y-%m-%d')) AS month_num,
    AVG(transactions) AS avg_transactions
FROM retail_forecasting.transactions
GROUP BY 1
ORDER BY 1
"""

monthly_avg = pd.read_sql(query, conn)

# Map month numbers to names
month_map = {
    1: "Jan", 2: "Feb", 3: "Mar",
    4: "Apr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Aug", 9: "Sep",
    10: "Oct", 11: "Nov", 12: "Dec"
}

monthly_avg["month"] = monthly_avg["month_num"].map(month_map)

plt.figure(figsize=(10,5))
plt.bar(monthly_avg["month"],
        monthly_avg["avg_transactions"],
       color="teal")

plt.title("Average Transactions by Month")
plt.xlabel("Month")
plt.ylabel("Average Transactions")
plt.show()


In [ ]:
query = """
SELECT 
    year(date_parse(date, '%Y-%m-%d')) AS year,
    month(date_parse(date, '%Y-%m-%d')) AS month_num,
    SUM(transactions) AS total_transactions
FROM retail_forecasting.transactions
GROUP BY 1, 2
ORDER BY 1, 2
"""

monthly_yearly = pd.read_sql(query, conn)

month_map = {
    1: "Jan", 2: "Feb", 3: "Mar",
    4: "Apr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Aug", 9: "Sep",
    10: "Oct", 11: "Nov", 12: "Dec"
}

monthly_yearly["month"] = monthly_yearly["month_num"].map(month_map)

plt.figure(figsize=(10,6))

for yr in monthly_yearly["year"].unique():
    data = monthly_yearly[monthly_yearly["year"] == yr]
    
    plt.plot(data["month"],
             data["total_transactions"],
             marker="o",
             label=str(yr))

plt.title("Yearly Seasonality Check")
plt.xlabel("Month")
plt.ylabel("Total Transactions")
plt.legend()
plt.show()


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt
import pandas as pd

query = """
SELECT 
    date,
    SUM(transactions) AS total_transactions
FROM retail_forecasting.transactions
GROUP BY date
ORDER BY date
"""

daily_total = pd.read_sql(query, conn)
daily_total["date"] = pd.to_datetime(daily_total["date"])

plt.figure(figsize=(10,5))

plot_acf(daily_total["total_transactions"], lags=60)

plt.xlabel("Lag (Days)")                     # X-axis = number of days back
plt.ylabel("Autocorrelation")                # Y-axis = correlation value
plt.title("Autocorrelation of Daily Transactions")

plt.show()


In [ ]:
query = """
SELECT 
    date,
    SUM(transactions) AS total_transactions
FROM retail_forecasting.transactions
GROUP BY date
ORDER BY date
"""

daily_total = pd.read_sql(query, conn)
daily_total["date"] = pd.to_datetime(daily_total["date"])

# Rolling Means
#daily_total["rolling_7"] = daily_total["total_transactions"].rolling(window=7).mean()
daily_total["rolling_30"] = daily_total["total_transactions"].rolling(window=30).mean()

plt.figure(figsize=(12,5))
plt.plot(daily_total["date"], daily_total["total_transactions"])
#plt.plot(daily_total["date"], daily_total["rolling_7"])
plt.plot(daily_total["date"], daily_total["rolling_30"])
plt.title("Daily Transactions with 30 Day Rolling Mean")
plt.xlabel("Date")
plt.ylabel("Total Transactions")
plt.show()


In [ ]:
##EDA for holidays_events table

In [ ]:
query = """
SELECT COUNT(*) AS total_records
FROM retail_forecasting.holidays_events
"""

pd.read_sql(query, conn)


In [ ]:
query = """
SELECT 
    MIN(date) AS min_date,
    MAX(date) AS max_date
FROM retail_forecasting.holidays_events
"""

pd.read_sql(query, conn)


In [ ]:
#Holidays Per Year
query = """
SELECT 
    year(date_parse(date, '%Y-%m-%d')) AS year,
    COUNT(*) AS holiday_count
FROM retail_forecasting.holidays_events
GROUP BY 1
ORDER BY 1
"""

holidays_per_year = pd.read_sql(query, conn)

plt.figure(figsize=(8,5))
plt.bar(holidays_per_year["year"],
        holidays_per_year["holiday_count"],
        color="royalblue")

plt.title("Number of Holidays per Year")
plt.xlabel("Year")
plt.ylabel("Holiday Count")
plt.show()


In [ ]:
#Holidays Per Month (Seasonality Check)
query = """
SELECT 
    month(date_parse(date, '%Y-%m-%d')) AS month_num,
    COUNT(*) AS holiday_count
FROM retail_forecasting.holidays_events
GROUP BY 1
ORDER BY 1
"""

holidays_per_month = pd.read_sql(query, conn)

month_map = {
    1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
    7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"
}

holidays_per_month["month"] = holidays_per_month["month_num"].map(month_map)

plt.figure(figsize=(10,5))
plt.bar(holidays_per_month["month"],
        holidays_per_month["holiday_count"],
        color="darkorange")

plt.title("Holidays per Month")
plt.xlabel("Month")
plt.ylabel("Holiday Count")
plt.show()


In [ ]:
#Holiday Types Distribution
query = """
SELECT type,
       COUNT(*) AS count
FROM retail_forecasting.holidays_events
GROUP BY type
ORDER BY count DESC
"""

holiday_types = pd.read_sql(query, conn)

plt.figure(figsize=(8,5))
plt.bar(holiday_types["type"],
        holiday_types["count"],
        color="forestgreen")

plt.title("Holiday Type Distribution")
plt.xlabel("Holiday Type")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()


In [ ]:
#Transferred Holidays
query = """
SELECT transferred,
       COUNT(*) AS count
FROM retail_forecasting.holidays_events
GROUP BY transferred
"""

pd.read_sql(query, conn)


In [ ]:
#National vs Local Holidays
query = """
SELECT locale,
       COUNT(*) AS count
FROM retail_forecasting.holidays_events
GROUP BY locale
"""

pd.read_sql(query, conn)


In [ ]:
#Distribution of locale
query = """
SELECT 
    locale,
    COUNT(*) AS count
FROM retail_forecasting.holidays_events
GROUP BY locale
ORDER BY count DESC
"""

locale_dist = pd.read_sql(query, conn)

plt.figure(figsize=(6,4))
plt.bar(locale_dist["locale"],
        locale_dist["count"],
        color="royalblue")

plt.title("Holiday Count by Locale")
plt.xlabel("Locale Type")
plt.ylabel("Count")
plt.show()


In [ ]:
#Distribution of locale_name
query = """
SELECT 
    locale_name,
    COUNT(*) AS count
FROM retail_forecasting.holidays_events
GROUP BY locale_name
ORDER BY count DESC
LIMIT 15
"""

locale_name_dist = pd.read_sql(query, conn)

plt.figure(figsize=(10,5))
plt.bar(locale_name_dist["locale_name"],
        locale_name_dist["count"],
        color="darkorange")

plt.title("Top 15 Locale Names by Holiday Count")
plt.xlabel("Locale Name")
plt.ylabel("Holiday Count")
plt.xticks(rotation=45)
plt.show()


In [ ]:
#Holidays by Locale per Year
query = """
SELECT 
    year(date_parse(date, '%Y-%m-%d')) AS year,
    locale,
    COUNT(*) AS count
FROM retail_forecasting.holidays_events
GROUP BY 1, 2
ORDER BY 1
"""

locale_year = pd.read_sql(query, conn)

plt.figure(figsize=(8,5))

for loc in locale_year["locale"].unique():
    data = locale_year[locale_year["locale"] == loc]
    plt.plot(data["year"],
             data["count"],
             marker="o",
             label=loc)

plt.title("Holidays per Year by Locale")
plt.xlabel("Year")
plt.ylabel("Holiday Count")
plt.legend()
plt.show()


In [ ]:
#Check National vs Non-National Ratio
query = """
SELECT 
    CASE 
        WHEN locale = 'National' THEN 'National'
        ELSE 'Non-National'
    END AS category,
    COUNT(*) AS count
FROM retail_forecasting.holidays_events
GROUP BY 1
"""

pd.read_sql(query, conn)


In [ ]:
##EDA for oil table

In [ ]:
query = """
SELECT *
FROM retail_forecasting.oil
ORDER BY date
LIMIT 5
"""

pd.read_sql(query, conn)


In [ ]:
#Check Missing Values
query = """
SELECT 
    COUNT(*) AS total_rows,
    COUNT(dcoilwtico) AS non_null_values
FROM retail_forecasting.oil
"""

pd.read_sql(query, conn)


In [ ]:
#Oil Price Trend Over Time
query = """
SELECT 
    date,
    dcoilwtico
FROM retail_forecasting.oil
ORDER BY date
"""

oil = pd.read_sql(query, conn)

plt.figure(figsize=(12,5))
plt.plot(pd.to_datetime(oil["date"]),
         oil["dcoilwtico"],
         color="darkgreen")

plt.title("Oil Price Over Time")
plt.xlabel("Date")
plt.ylabel("Oil Price")
plt.show()


In [ ]:
#Yearly Average Oil Price (Check Trend)
query = """
SELECT 
    year(date_parse(date, '%Y-%m-%d')) AS year,
    AVG(dcoilwtico) AS avg_price
FROM retail_forecasting.oil
GROUP BY 1
ORDER BY 1
"""

yearly_oil = pd.read_sql(query, conn)

plt.figure(figsize=(8,4))
plt.plot(yearly_oil["year"],
         yearly_oil["avg_price"],
         marker="o",
         color="crimson")

plt.title("Yearly Average Oil Price")
plt.xlabel("Year")
plt.ylabel("Average Oil Price")
plt.show()


In [ ]:
#Monthly Average Oil Price
query = """
SELECT 
    month(date_parse(date, '%Y-%m-%d')) AS month,
    AVG(dcoilwtico) AS avg_price
FROM retail_forecasting.oil
GROUP BY 1
ORDER BY 1
"""

monthly_oil = pd.read_sql(query, conn)

month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

monthly_oil["month_name"] = monthly_oil["month"].apply(lambda x: month_labels[x-1])

plt.figure(figsize=(8,4))
plt.plot(monthly_oil["month_name"],
         monthly_oil["avg_price"],
         marker="o",
         color="purple")

plt.title("Average Oil Price by Month")
plt.xlabel("Month")
plt.ylabel("Average Oil Price")
plt.show()


In [ ]:
#Rolling 30-Day Mean (Smooth Trend)
oil["date"] = pd.to_datetime(oil["date"])
oil = oil.sort_values("date")

oil["rolling_30"] = oil["dcoilwtico"].rolling(30).mean()

plt.figure(figsize=(12,5))
plt.plot(oil["date"], oil["dcoilwtico"], alpha=0.4, label="Daily Price")
plt.plot(oil["date"], oil["rolling_30"], linewidth=2, label="30-Day Rolling Mean")

plt.title("Oil Price with 30-Day Rolling Mean")
plt.xlabel("Date")
plt.ylabel("Oil Price")
plt.legend()
plt.show()
